# Huber Loss en profundidad: precisión normal sin obedecer a los outliers

Huber Loss es útil cuando la mayoría de los datos son razonables, pero unos pocos valores extremos son errores de medición, errores de digitación o situaciones tan raras que no deberían dictar el comportamiento normal del modelo.

La pregunta que responderemos es: **¿cómo puede Huber mantener una recta precisa para los casos habituales sin ignorar por completo los casos extraños?**

## 1. Antes de Huber: tres maneras de castigar un error

Un **residuo** es $r = y - \hat y$: cuánto se alejó la predicción del valor real. Las pérdidas deciden cuánto duele ese error durante el entrenamiento.

| Pérdida | Fórmula para un residuo $r$ | Reacción a un error muy grande |
| --- | --- | --- |
| MAE | $|r|$ | Crece de forma lineal. Es robusta, pero no distingue tanto entre errores moderados y grandes. |
| MSE | $r^2$ | Crece cuadráticamente. Un outlier puede dominar la recta. |
| Huber | cuadrática cerca de 0; lineal lejos de 0 | Combina la precisión local de MSE con la robustez de MAE. |

Importante: no existe una pérdida “mejor” en todos los problemas. Huber es preferible si el objetivo es modelar el patrón típico y los extremos no son una señal que se deba aprender.

## 2. La fórmula y el umbral $\delta$

Huber usa un punto de cambio llamado $\delta$:

$$L_\delta(r) = \begin{cases}
\frac{1}{2}r^2 & \text{si } |r| \leq \delta \\n
\delta\left(|r|-\frac{1}{2}\delta\right) & \text{si } |r| > \delta
\end{cases}$$

- Dentro de $[-\delta, \delta]$, se comporta como MSE: pequeños ajustes importan y la recta queda afinada.
- Fuera de ese intervalo, se comporta como MAE: el castigo deja de crecer al cuadrado.

Si predices precios en dólares, $\delta$ está relacionado con dólares; si predices minutos, con minutos. Elegirlo exige conocer qué error todavía consideras normal.

In [1]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from scipy.special import huber
from sklearn.datasets import load_diabetes
from sklearn.linear_model import HuberRegressor, LinearRegression, QuantileRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

delta = 10.0
residuos = np.linspace(-50, 50, 501)
perdidas = pl.DataFrame({
    "residuo": np.concatenate([residuos, residuos, residuos]),
    "perdida": np.concatenate([residuos**2 / 2, np.abs(residuos), huber(delta, residuos)]),
    "tipo": (["MSE / 2"] * len(residuos) + ["MAE"] * len(residuos) + ["Huber (delta=10)"] * len(residuos)),
})

fig = px.line(
    perdidas, x="residuo", y="perdida", color="tipo",
    title="Huber cambia de curva cuadrática a recta en ±delta",
)
fig.add_vline(x=-delta, line_dash="dash", line_color="gray")
fig.add_vline(x=delta, line_dash="dash", line_color="gray")
fig.show()

Lee la gráfica de izquierda a derecha: cerca de cero, Huber sigue a la curva de MSE; lejos de cero, se acerca al comportamiento lineal de MAE. El valor del outlier sigue contando, pero pierde el poder de arrastrar toda la recta hacia él.

En este notebook usamos `scipy.special.huber` para calcular la pérdida por residuo. Para entrenar, usaremos `sklearn.linear_model.HuberRegressor`, que implementa la optimización robusta; no escribiremos un optimizador propio.

## 3. Caso práctico: predicción de alquiler mensual

Una inmobiliaria quiere estimar el alquiler mensual a partir de los metros cuadrados. La mayoría de sus registros son correctos. Sin embargo, una persona escribió **12 000** en lugar de **1 200** para un departamento de 55 m².

El objetivo es estimar el alquiler habitual de nuevos departamentos, no repetir ese error de digitación. Esta es una situación donde Huber suele ser mejor que MSE: MSE intentará acomodar el valor 12 000; Huber limitará su influencia. MAE también será robusta, pero Huber puede ajustar más finamente la mayoría de errores pequeños.

In [2]:
# Alquileres mensuales en dólares. El último registro es un error de captura conocido.
entrenamiento = pl.DataFrame({
    "metros_cuadrados": [35, 42, 48, 55, 62, 70, 78, 86, 95, 55],
    "alquiler_usd": [820, 960, 1080, 1210, 1360, 1530, 1710, 1880, 2080, 12000],
    "registro": ["normal"] * 9 + ["error de digitación"],
})

fig = px.scatter(
    entrenamiento, x="metros_cuadrados", y="alquiler_usd", color="registro",
    title="Un solo dato mal registrado está muy lejos del patrón normal",
    color_discrete_map={"normal": "#1f77b4", "error de digitación": "#d62728"},
)
fig.show()
entrenamiento

metros_cuadrados,alquiler_usd,registro
i64,i64,str
35,820,"""normal"""
42,960,"""normal"""
48,1080,"""normal"""
55,1210,"""normal"""
62,1360,"""normal"""
70,1530,"""normal"""
78,1710,"""normal"""
86,1880,"""normal"""
95,2080,"""normal"""


## 4. Entrenamos tres modelos sobre los mismos datos

- `LinearRegression` minimiza MSE.
- `QuantileRegressor(quantile=0.5)` minimiza una pérdida equivalente a MAE; predice la mediana condicional.
- `HuberRegressor` usa Huber Loss. Su parámetro `epsilon` cumple un papel de tolerancia al outlier, pero trabaja sobre una escala interna robusta; no es exactamente el mismo $\delta$ en dólares de la gráfica anterior.

Los datos se manipulan con Polars; se convierten a NumPy solo al pasarlos a las APIs de scikit-learn.

In [3]:
X_train = entrenamiento.select("metros_cuadrados").to_numpy()
y_train = entrenamiento["alquiler_usd"].to_numpy()

modelo_mse = LinearRegression().fit(X_train, y_train)
modelo_mae = QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs").fit(X_train, y_train)
modelo_huber = HuberRegressor(epsilon=1.35, max_iter=1_000).fit(X_train, y_train)

modelos = {
    "MSE: LinearRegression": modelo_mse,
    "MAE: QuantileRegressor": modelo_mae,
    "Huber: HuberRegressor": modelo_huber,
}

pl.DataFrame({
    "modelo": list(modelos),
    "w (USD por m²)": [modelo.coef_[0] for modelo in modelos.values()],
    "b (USD)": [modelo.intercept_ for modelo in modelos.values()],
})

modelo,w (USD por m²),b (USD)
str,f64,f64
"""MSE: LinearRegression""",-3.039596,2653.278698
"""MAE: QuantileRegressor""",20.909091,81.818182
"""Huber: HuberRegressor""",20.965124,75.434019


In [4]:
metros_linea = np.linspace(30, 100, 200).reshape(-1, 1)
fig = px.scatter(
    entrenamiento, x="metros_cuadrados", y="alquiler_usd", color="registro",
    title="La recta MSE es arrastrada por el error; Huber protege el patrón normal",
    color_discrete_map={"normal": "#1f77b4", "error de digitación": "#d62728"},
)
colores = ["#ff7f0e", "#2ca02c", "#9467bd"]
for (nombre, modelo), color in zip(modelos.items(), colores):
    fig.add_trace(go.Scatter(
        x=metros_linea.ravel(), y=modelo.predict(metros_linea), mode="lines",
        name=nombre, line={"color": color},
    ))
fig.show()

## 5. ¿Cuál generaliza mejor a departamentos normales?

Evaluamos los tres modelos sobre departamentos nuevos y correctamente registrados. Esta separación es crucial: evaluar con los mismos datos corruptos de entrenamiento premiaría injustamente al modelo que persigue el error de digitación.

In [5]:
prueba_limpia = pl.DataFrame({
    "metros_cuadrados": [38, 50, 60, 72, 84, 98],
    "alquiler_usd": [880, 1120, 1330, 1580, 1850, 2160],
})
X_test = prueba_limpia.select("metros_cuadrados").to_numpy()
y_test = prueba_limpia["alquiler_usd"].to_numpy()

resultados = pl.DataFrame({
    "modelo": list(modelos),
    "MAE en prueba (USD)": [mean_absolute_error(y_test, modelo.predict(X_test)) for modelo in modelos.values()],
    "RMSE en prueba (USD)": [root_mean_squared_error(y_test, modelo.predict(X_test)) for modelo in modelos.values()],
    "Huber promedio en prueba": [np.mean(huber(100.0, y_test - modelo.predict(X_test))) for modelo in modelos.values()],
}).sort("RMSE en prueba (USD)")
resultados

modelo,MAE en prueba (USD),RMSE en prueba (USD),Huber promedio en prueba
str,f64,f64,f64
"""MAE: QuantileRegressor""",10.909091,13.816986,95.454545
"""Huber: HuberRegressor""",10.554216,14.097594,99.371076
"""MSE: LinearRegression""",962.959112,1082.153756,91295.911174


En esta prueba limpia, Huber debería quedar cerca del mejor resultado porque aprende la tendencia de los nueve registros normales y reduce el efecto del registro absurdo. MAE también debería resistir bien, aunque puede ser menos estable o preciso cuando los errores normales son pequeños. MSE suele quedar peor porque su recta intenta reducir un error de miles de dólares.

Esto no significa que debamos ocultar todos los outliers. Primero hay que investigarlos: quizá el departamento de 12 000 USD sea un penthouse real y necesite variables nuevas como barrio, lujo o número de habitaciones. Huber es apropiada aquí porque **sabemos** que el valor fue un error de digitación.

## 6. Guía corta para elegir

- Elige **MSE/RMSE** si los errores grandes son genuinamente más dañinos y deben mandar en la optimización.
- Elige **MAE** si quieres medir el error típico en unidades claras y tratar cada minuto, dólar o unidad de error de forma proporcional.
- Prueba **Huber** cuando los outliers son pocos, no representan la población que quieres predecir y todavía quieres que los errores pequeños se ajusten con precisión.

No elijas una pérdida solo porque produzca el número más bajo en entrenamiento. Define primero qué errores importan en el mundo real, usa un conjunto de prueba representativo y compara las métricas que correspondan a esa decisión.

## 7. Ejemplo con un dataset real: progresión de diabetes

Ahora usamos `load_diabetes`, un dataset de prueba real incluido en scikit-learn. Contiene mediciones clínicas de 442 personas y una variable objetivo continua: una medida de progresión de la enfermedad un año después.

El dataset no afirma que tenga errores extremos de digitación. Para estudiar Huber de forma controlada, haremos algo parecido a un incidente común en proyectos reales: **contaminaremos solo 18 etiquetas del entrenamiento** con un error artificial de ±500. La prueba se conserva limpia. Así sabemos que el modelo correcto es el que aprende de la mayoría de registros sanos y no el que memoriza las etiquetas dañadas.

In [ ]:
datos_diabetes = load_diabetes()
# Elegimos solo BMI para poder representar cada modelo como una recta.
# En un proyecto real podríamos usar las diez variables clínicas.
indice_bmi = list(datos_diabetes.feature_names).index("bmi")
X = datos_diabetes.data[:, [indice_bmi]]
y = datos_diabetes.target
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.30, random_state=42,
)

# Simulamos un problema de captura solo en entrenamiento, sin tocar la prueba.
generador = np.random.default_rng(42)
y_entrenamiento_contaminado = y_entrenamiento.copy()
indices_outlier = generador.choice(len(y_entrenamiento), size=18, replace=False)
y_entrenamiento_contaminado[indices_outlier] += generador.choice(
    [-500, 500], size=len(indices_outlier),
)

pl.DataFrame({
    "conjunto": ["entrenamiento", "prueba limpia"],
    "filas": [len(y_entrenamiento), len(y_prueba)],
    "outliers artificiales": [len(indices_outlier), 0],
})

### Tres modelos, misma información

Para que podamos dibujar rectas, aquí usamos únicamente `bmi` como variable de entrada. Los tres reciben exactamente la misma información y las mismas etiquetas contaminadas. Usamos `StandardScaler` dentro de un `Pipeline`: estandariza la variable usando únicamente el entrenamiento y evita fuga de información hacia la prueba.

- `LinearRegression` minimiza MSE y, por ello, siente los 18 errores enormes con mucha fuerza.
- `QuantileRegressor` con cuantil 0.5 usa una pérdida equivalente a MAE.
- `HuberRegressor` es cuadrático para la mayoría de residuos y reduce la influencia de los 18 contaminados.

In [ ]:
modelos_diabetes = {
    "MSE: LinearRegression": make_pipeline(StandardScaler(), LinearRegression()),
    "MAE: QuantileRegressor": make_pipeline(
        StandardScaler(), QuantileRegressor(quantile=0.5, alpha=0.0, solver="highs"),
    ),
    "Huber: HuberRegressor": make_pipeline(
        StandardScaler(), HuberRegressor(epsilon=1.35, max_iter=1_000),
    ),
}

for modelo in modelos_diabetes.values():
    modelo.fit(X_entrenamiento, y_entrenamiento_contaminado)

resultados_diabetes = pl.DataFrame({
    "modelo": list(modelos_diabetes),
    "MAE en prueba limpia": [
        mean_absolute_error(y_prueba, modelo.predict(X_prueba))
        for modelo in modelos_diabetes.values()
    ],
    "RMSE en prueba limpia": [
        root_mean_squared_error(y_prueba, modelo.predict(X_prueba))
        for modelo in modelos_diabetes.values()
    ],
}).sort("RMSE en prueba limpia")
resultados_diabetes

### Las rectas que aprende cada pérdida

La gráfica siguiente hace visible la diferencia. Los puntos rojos son las etiquetas contaminadas que el modelo vio durante el entrenamiento. La recta naranja de MSE intenta acercarse a ellos porque sus errores al cuadrado pesan muchísimo. La recta Huber puede seguir más de cerca la nube azul de casos normales.

In [ ]:
es_outlier = np.zeros(len(y_entrenamiento), dtype=bool)
es_outlier[indices_outlier] = True
puntos_entrenamiento = pl.DataFrame({
    "bmi_normalizado": X_entrenamiento.ravel(),
    "progresion_entrenamiento": y_entrenamiento_contaminado,
    "tipo": np.where(es_outlier, "etiqueta contaminada", "registro normal"),
})

bmi_linea = np.linspace(X_entrenamiento.min(), X_entrenamiento.max(), 200).reshape(-1, 1)
fig = px.scatter(
    puntos_entrenamiento, x="bmi_normalizado", y="progresion_entrenamiento", color="tipo",
    title="Rectas entrenadas con BMI y etiquetas contaminadas",
    color_discrete_map={"registro normal": "#1f77b4", "etiqueta contaminada": "#d62728"},
)
colores = ["#ff7f0e", "#2ca02c", "#9467bd"]
for (nombre, modelo), color in zip(modelos_diabetes.items(), colores):
    fig.add_trace(go.Scatter(
        x=bmi_linea.ravel(), y=modelo.predict(bmi_linea), mode="lines",
        name=nombre, line={"color": color, "width": 3},
    ))
fig.update_layout(
    xaxis_title="BMI normalizado",
    yaxis_title="Progresión de diabetes al año",
)
fig.show()

In [ ]:
resultados_largos = resultados_diabetes.unpivot(
    index="modelo",
    on=["MAE en prueba limpia", "RMSE en prueba limpia"],
    variable_name="métrica",
    value_name="error",
)
px.bar(
    resultados_largos, x="modelo", y="error", color="métrica", barmode="group",
    title="El modelo Huber generaliza mejor cuando el entrenamiento tiene outliers de captura",
).show()

### Qué demuestra este experimento

En la prueba limpia, Huber mejora claramente el RMSE frente a MSE y queda muy cerca de MAE; el orden exacto puede cambiar según la muestra, la contaminación y el valor de `epsilon`. MSE fue afectado por los errores de ±500 y movió sus parámetros para intentar reducirlos. MAE resiste esos casos, mientras que Huber conserva información cuadrática de los residuos normales.

La lección no es “agrega outliers para que Huber gane”. Es esta: cuando un proceso real puede dañar una pequeña parte de las etiquetas y tu objetivo es predecir nuevos casos normales, prueba un modelo robusto y evalúalo contra datos de prueba que representen el uso real. Si los extremos son diagnósticos importantes y son correctos, investigarlos o añadir variables puede ser mejor que reducir su influencia.